In [1]:
import torch.nn as nn

In [2]:
class mymodel(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Linear(64,16),
            nn.ReLU(),
            nn.Linear(16,1)
        )

    def forward(self,x):
        x=self.layers(x)
        x = x.squeeze(1)
        return x

In [3]:
model=mymodel(input_dim=95)
print(model)

mymodel(
  (layers): Sequential(
    (0): Linear(in_features=95, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [4]:
from torch.utils.data import Dataset 
import torch 
import pandas as pd

class COVID19Dataset(Dataset):
    def __init__(self,x,y=None):
        super().__init__()
        if y is None:
            self.y=y
        else:
            self.y=torch.FloatTensor(y)
        self.x=torch.FloatTensor(x)

    def __len__(self):
        return len(self.x)

    def __getitem__(self,index):
        if self.y is None:
            return self.x[index]
        else:
            return self.x[index],self.y[index]

df=pd.read_csv('./covid.train.csv')
df.head()

,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,work_outside_home.4,shop.4,restaurant.4,spent_time.4,large_event.4,public_transit.4,anxious.4,depressed.4,worried_finances.4,tested_positive.4
0,0,0,0,0,0,0,0,0,1,0,...,31.113209,67.394551,36.674291,40.743132,17.842221,4.093712,10.440071,8.627117,37.329512,7.456154
1,1,0,0,0,0,0,1,0,0,0,...,33.920257,64.398380,34.612238,44.035688,17.808103,4.924935,10.172662,9.954333,32.508881,8.010957
2,2,0,0,0,0,0,0,0,0,0,...,31.604604,62.101064,26.521875,36.746453,13.903667,7.313833,10.388712,7.956139,36.745588,2.906977
3,3,0,0,0,0,0,0,0,0,0,...,35.115738,67.935520,38.022492,48.434809,27.134876,3.101904,10.498683,8.231522,38.680162,12.575816
4,4,0,0,0,0,0,0,0,0,0,...,35.129714,69.934592,38.242368,49.095933,22.683709,4.594620,9.878927,9.469290,28.344123,21.428589


In [5]:
data=df.values
x_data = data[:,1:-1]
y_data = data[:,-1]
train_dataset = COVID19Dataset(x=x_data, y=y_data)
print(f"成功打包了 {len(train_dataset)} 条数据！")

成功打包了 2699 条数据！


In [6]:
from torch.utils.data import DataLoader
import torch.optim as optim

In [7]:
batch_size=60
train_loader=DataLoader(train_dataset,batch_size,shuffle=True)

In [8]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)
model=mymodel(input_dim=x_data.shape[1]).to(device)

criterion=nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

cpu


In [9]:
epochs=100

print("🚀 训练开始！")
for epoch in range(epochs):
    model.train()
    total_loss=0

    for x_batch,y_batch in train_loader:
        x_batch,y_batch=x_batch.to(device),y_batch.to(device)

        optimizer.zero_grad()
        pred=model(x_batch)
        loss=criterion(pred,y_batch)
        loss.backward()
        optimizer.step()

        total_loss+=loss

    if (epoch + 1) % 10 == 0:
        avg_loss = total_loss / len(train_loader)
        print(f"第 {epoch+1:3d}/{epochs} 圈, 平均误差 Loss: {avg_loss:.4f}")

print("✅ 训练完成！")

🚀 训练开始！
第  10/100 圈, 平均误差 Loss: 2.4373
第  20/100 圈, 平均误差 Loss: 2.1009
第  30/100 圈, 平均误差 Loss: 1.5790
第  40/100 圈, 平均误差 Loss: 1.4071
第  50/100 圈, 平均误差 Loss: 1.2953
第  60/100 圈, 平均误差 Loss: 1.1791
第  70/100 圈, 平均误差 Loss: 1.2535
第  80/100 圈, 平均误差 Loss: 1.1372
第  90/100 圈, 平均误差 Loss: 1.2754
第 100/100 圈, 平均误差 Loss: 1.1757
✅ 训练完成！


In [10]:
import numpy as np 
test_df=pd.read_csv("./covid.test.csv")
test_df.head()


,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,travel_outside_state.4,work_outside_home.4,shop.4,restaurant.4,spent_time.4,large_event.4,public_transit.4,anxious.4,depressed.4,worried_finances.4
0,0,0,0,0,0,0,0,0,0,0,...,18.463415,32.470622,65.236954,31.730028,42.031183,18.188452,3.472105,12.452570,9.400024,30.550873
1,1,0,0,0,0,0,0,0,0,0,...,24.201702,31.499036,60.360030,28.319417,39.919723,16.614292,5.992332,11.747528,7.438606,35.364871
2,2,0,0,0,0,0,0,0,0,0,...,13.742657,28.696996,66.114428,31.211381,40.448538,16.998722,13.951631,12.898089,10.214711,37.028705
3,3,0,0,0,0,0,0,0,0,0,...,13.763442,27.907935,66.581525,29.112434,41.123571,13.949904,5.337427,14.944972,11.556292,32.146346
4,4,0,0,0,0,0,0,0,0,0,...,14.427636,37.623197,66.502657,29.716613,41.790256,18.855524,3.015219,15.225684,10.126752,39.163694


In [11]:
test_data=test_df.values
x_test=test_data[:,1:]
print(x_test.shape)
test_dataset=COVID19Dataset(x=x_test,y=None)
test_dataloader=DataLoader(test_dataset,batch_size=270,shuffle=False)

print("📝 开始期末考试...")
preds=[]
with torch.no_grad():
    for x_batch in test_dataloader:
        x_batch=x_batch.to(device)
        pred=model(x_batch)
        preds.append(pred.detach().cpu().numpy())

preds=np.concatenate(preds)

submission=pd.DataFrame({ 
    "id":test_df["id"],
    "test_positive": preds
})
submission.to_csv("my_first_ans.csv",index=False)
print("🎉 答题卡已生成！快去文件夹里看看 my_first_submission.csv 吧！")

(1078, 116)
📝 开始期末考试...
🎉 答题卡已生成！快去文件夹里看看 my_first_submission.csv 吧！
